# Newt - multitask fine-tuning


In [ ]:
from pathlib import Path
import json
import shutil
import tempfile
import urllib.request
import zipfile
from kaggle_secrets import UserSecretsClient

REPOSITORY = 'leolazzz/newt-coursework'
REVISION = 'main'
project = Path('/kaggle/working/newt-coursework')
token = UserSecretsClient().get_secret('GITHUB_TOKEN')
headers = {'Authorization': f'Bearer {token}', 'User-Agent': 'newt-coursework'}
api = f'https://api.github.com/repos/{REPOSITORY}'

with urllib.request.urlopen(urllib.request.Request(
        f'{api}/commits/{REVISION}', headers=headers)) as response:
    commit = json.load(response)['sha']

Path('/kaggle/temp').mkdir(parents=True, exist_ok=True)

with tempfile.TemporaryDirectory(dir='/kaggle/temp') as temporary:
    temporary = Path(temporary)
    archive_path = temporary / 'source.zip'
    with urllib.request.urlopen(urllib.request.Request(
            f'{api}/zipball/{commit}', headers=headers)) as response:
        with archive_path.open('wb') as output:
            shutil.copyfileobj(response, output)
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(temporary / 'source')
    source = next((temporary / 'source').iterdir())
    shutil.copytree(source, project, dirs_exist_ok=True)

(project / 'project-commit.txt').write_text(commit + '\n')
del token, headers
print('GitHub version:', commit)


In [ ]:
%%bash
set -e
bash /kaggle/working/newt-coursework/scripts/setup.sh


In [ ]:
%%bash
set -eo pipefail
export CUDA_VISIBLE_DEVICES=0
export PYTHONUNBUFFERED=1
unset PYTHONPATH
cd /kaggle/working/newt-coursework
/kaggle/temp/newt-isolated/python-env/bin/python train.py 2>&1 | tee run.log


In [ ]:
pointer = json.loads((project / 'latest-run.json').read_text())
folder = Path(pointer['folder'])
print((folder / 'status.json').read_text())
print((folder / 'summary.csv').read_text())
